In [ ]:
!nvidia-smi || true

!pip install -q tensorflow numpy scikit-learn tqdm

Thu May  7 17:50:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from pathlib import Path

# Training size
SAMPLES_PER_CLASS = 15000
MAX_LEN = 256
BATCH_SIZE = 256
EPOCHS = 40
SEED = 42

# Output paths
DATA_DIR = Path("/content/data")
NDJSON_DIR = DATA_DIR / "quickdraw_simplified"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = Path("/content/quickdraw_stroke_tflite")

NDJSON_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Start with clean, visually distinct categories.
CLASSES = [
    "The Mona Lisa",
    "apple",
    "banana",
    "cat",
    "dog",
    "fish",
    "bird",
    "airplane",
    "car",
    "bicycle",
    "bus",
    "tree",
    "flower",
    "house",
    "chair",
    "table",
    "cup",
    "fork",
    "umbrella",
    "star",
    "moon",
    "sun",
    "cloud",
    "crown",
    "pizza",
    "ice cream",
    "book",
    "clock",
    "eye",
    "face",
]

print("Classes:", len(CLASSES))

Classes: 30


In [ ]:
import subprocess
from pathlib import Path

for class_name in CLASSES:
    src = f"gs://quickdraw_dataset/full/simplified/{class_name}.ndjson"
    dst = NDJSON_DIR / f"{class_name}.ndjson"

    if dst.exists():
        print(f"Already exists: {dst}")
        continue

    print(f"Downloading: {class_name}")
    subprocess.run(["gsutil", "cp", src, str(dst)], check=True)

print("Download complete.")

Downloading: The Mona Lisa
Downloading: apple
Downloading: banana
Downloading: cat
Downloading: dog
Downloading: fish
Downloading: bird
Downloading: airplane
Downloading: car
Downloading: bicycle
Downloading: bus
Downloading: tree
Downloading: flower
Downloading: house
Downloading: chair
Downloading: table
Downloading: cup
Downloading: fork
Downloading: umbrella
Downloading: star
Downloading: moon
Downloading: sun
Downloading: cloud
Downloading: crown
Downloading: pizza
Downloading: ice cream
Downloading: book
Downloading: clock
Downloading: eye
Downloading: face
Download complete.


In [ ]:
import json
import random
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

random.seed(SEED)
np.random.seed(SEED)


def normalize_strokes(drawing):
    """
    QuickDraw simplified drawing format:
    [
      [[x0, x1, ...], [y0, y1, ...]],
      [[x0, x1, ...], [y0, y1, ...]]
    ]

    Returns a list of strokes:
    [
      [(x, y), (x, y), ...],
      [(x, y), (x, y), ...]
    ]
    """
    points = []

    for stroke in drawing:
        if len(stroke) != 2:
            continue

        xs, ys = stroke

        for x, y in zip(xs, ys):
            points.append((float(x), float(y)))

    if not points:
        return []

    xs_all = [p[0] for p in points]
    ys_all = [p[1] for p in points]

    min_x, max_x = min(xs_all), max(xs_all)
    min_y, max_y = min(ys_all), max(ys_all)

    width = max(max_x - min_x, 1.0)
    height = max(max_y - min_y, 1.0)

    scale = 255.0 / max(width, height)

    normalized = []

    for stroke in drawing:
        if len(stroke) != 2:
            continue

        xs, ys = stroke
        new_stroke = []

        for x, y in zip(xs, ys):
            nx = (float(x) - min_x) * scale
            ny = (float(y) - min_y) * scale
            new_stroke.append((nx, ny))

        if len(new_stroke) >= 2:
            normalized.append(new_stroke)

    return normalized


def strokes_to_sequence(drawing, max_len=MAX_LEN):
    """
    Converts QuickDraw strokes into a fixed tensor:
    [max_len, 5]

    Each row:
    [dx, dy, pen_down, pen_up, end]
    """
    strokes = normalize_strokes(drawing)

    seq = []
    prev_x = None
    prev_y = None

    for stroke in strokes:
        for i, (x, y) in enumerate(stroke):
            if prev_x is None:
                dx = 0.0
                dy = 0.0
            else:
                dx = x - prev_x
                dy = y - prev_y

            is_last_point = i == len(stroke) - 1

            pen_down = 0.0 if is_last_point else 1.0
            pen_up = 1.0 if is_last_point else 0.0
            end = 0.0

            seq.append([
                dx / 255.0,
                dy / 255.0,
                pen_down,
                pen_up,
                end,
            ])

            prev_x = x
            prev_y = y

            if len(seq) >= max_len - 1:
                break

        if len(seq) >= max_len - 1:
            break

    # End token
    seq.append([0.0, 0.0, 0.0, 0.0, 1.0])

    # Pad
    while len(seq) < max_len:
        seq.append([0.0, 0.0, 0.0, 0.0, 0.0])

    return np.array(seq[:max_len], dtype=np.float32)


def read_class_samples(class_name, label_id, limit):
    path = NDJSON_DIR / f"{class_name}.ndjson"

    xs = []
    ys = []

    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Processing {class_name}"):
            row = json.loads(line)

            # Keep only drawings Google recognized.
            # Later, you can remove this for more variation.
            if not row.get("recognized", False):
                continue

            drawing = row.get("drawing")
            if not drawing:
                continue

            x = strokes_to_sequence(drawing)

            xs.append(x)
            ys.append(label_id)

            if len(xs) >= limit:
                break

    return xs, ys


all_x = []
all_y = []

label_to_id = {name: i for i, name in enumerate(CLASSES)}

for class_name in CLASSES:
    label_id = label_to_id[class_name]
    xs, ys = read_class_samples(class_name, label_id, SAMPLES_PER_CLASS)

    all_x.extend(xs)
    all_y.extend(ys)

x = np.stack(all_x).astype(np.float32)
y = np.array(all_y, dtype=np.int64)

print("X:", x.shape)
print("Y:", y.shape)
print("Classes:", len(CLASSES))

Processing The Mona Lisa: 16284it [00:03, 5323.40it/s]
Processing apple: 15538it [00:02, 6128.42it/s]
Processing banana: 16152it [00:02, 6597.42it/s]
Processing cat: 17918it [00:04, 4145.95it/s]
Processing dog: 15895it [00:03, 5251.01it/s]
Processing fish: 15897it [00:02, 6290.33it/s]
Processing bird: 17876it [00:02, 6271.59it/s]
Processing airplane: 16737it [00:03, 4321.33it/s]
Processing car: 17043it [00:02, 5924.24it/s]
Processing bicycle: 15455it [00:02, 5229.79it/s]
Processing bus: 17308it [00:03, 5685.46it/s]
Processing tree: 15709it [00:04, 3474.22it/s]
Processing flower: 15651it [00:03, 5193.29it/s]
Processing house: 15336it [00:02, 6101.85it/s]
Processing chair: 16397it [00:03, 4512.80it/s]
Processing table: 15716it [00:03, 4623.19it/s]
Processing cup: 16187it [00:02, 6129.12it/s]
Processing fork: 15872it [00:02, 6486.30it/s]
Processing umbrella: 15465it [00:02, 5958.13it/s]
Processing star: 15543it [00:03, 4340.40it/s]
Processing moon: 15906it [00:02, 6118.60it/s]
Processing 

X: (450000, 256, 5)
Y: (450000,)
Classes: 30


In [ ]:
x_train, x_temp, y_train, y_temp = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

print("Train:", x_train.shape, y_train.shape)
print("Val:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

np.save(PROCESSED_DIR / "x_train.npy", x_train)
np.save(PROCESSED_DIR / "y_train.npy", y_train)
np.save(PROCESSED_DIR / "x_val.npy", x_val)
np.save(PROCESSED_DIR / "y_val.npy", y_val)
np.save(PROCESSED_DIR / "x_test.npy", x_test)
np.save(PROCESSED_DIR / "y_test.npy", y_test)

with open(PROCESSED_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, indent=2)

Train: (360000, 256, 5) (360000,)
Val: (45000, 256, 5) (45000,)
Test: (45000, 256, 5) (45000,)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(CLASSES)

inputs = keras.Input(shape=(MAX_LEN, 5), name="stroke_input")

x_in = layers.Conv1D(64, kernel_size=5, padding="same", activation="relu")(inputs)
x_in = layers.BatchNormalization()(x_in)
x_in = layers.MaxPooling1D(pool_size=2)(x_in)

x_in = layers.Conv1D(128, kernel_size=5, padding="same", activation="relu")(x_in)
x_in = layers.BatchNormalization()(x_in)
x_in = layers.MaxPooling1D(pool_size=2)(x_in)

x_in = layers.Conv1D(256, kernel_size=3, padding="same", activation="relu")(x_in)
x_in = layers.BatchNormalization()(x_in)

x_in = layers.GlobalAveragePooling1D()(x_in)

x_in = layers.Dense(256, activation="relu")(x_in)
x_in = layers.Dropout(0.30)(x_in)

outputs = layers.Dense(num_classes, activation="softmax", name="class_probs")(x_in)

model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
    ],
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ stroke_input (InputLayer)       │ (None, 256, 5)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 256, 64)        │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 128, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 128, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 64, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_probs (Dense)             │ (None, 30)             │         7,710 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 216,606 (846.12 KB)

 Trainable params: 215,710 (842.62 KB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "best.keras"),
        monitor="val_top3",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_top3",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_top3",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1,
    ),
]

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(MODEL_DIR / "final.keras")

with open(MODEL_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, indent=2)

print("Saved Keras model and labels.")

Epoch 1/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6200 - loss: 1.2890 - top3: 0.8055
Epoch 1: val_top3 improved from None to 0.76780, saving model to /content/quickdraw_stroke_tflite/best.keras

Epoch 1: finished saving model to /content/quickdraw_stroke_tflite/best.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 44s 24ms/step - accuracy: 0.7771 - loss: 0.7453 - top3: 0.9143 - val_accuracy: 0.4638 - val_loss: 2.7756 - val_top3: 0.7678 - learning_rate: 0.0010
Epoch 2/40
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9061 - loss: 0.3063 - top3: 0.9807
Epoch 2: val_top3 improved from 0.76780 to 0.95491, saving model to /content/quickdraw_stroke_tflite/best.keras

Epoch 2: finished saving model to /content/quickdraw_stroke_tflite/best.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - accuracy: 0.9113 - loss: 0.2891 - top3: 0.9820 - val_accuracy: 0.8161 - val_loss: 0.5818 - val_top3: 0.9549 - learning_rate: 0.0010
Epoch 3/40
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step 

In [ ]:
best_model = keras.models.load_model(MODEL_DIR / "best.keras")

loss, acc, top3 = best_model.evaluate(x_test, y_test, batch_size=BATCH_SIZE)

print("Test loss:", loss)
print("Test top-1 accuracy:", acc)
print("Test top-3 accuracy:", top3)

176/176 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9537 - loss: 0.1726 - top3: 0.9926
Test loss: 0.17257703840732574
Test top-1 accuracy: 0.9537333250045776
Test top-3 accuracy: 0.9926000237464905


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()

tflite_path = MODEL_DIR / "quickdraw_stroke_model_float32.tflite"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Saved:", tflite_path)
print("Size MB:", tflite_path.stat().st_size / 1024 / 1024)

Saved artifact at '/tmp/tmpfhele317'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 5), dtype=tf.float32, name='stroke_input')
Output Type:
  TensorSpec(shape=(None, 30), dtype=tf.float32, name=None)
Captures:
  137584927006672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578389776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578389392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578386704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578389200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578388240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578389008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578390160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578388048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578387088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578386128: T

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_float16 = converter.convert()

tflite_float16_path = MODEL_DIR / "quickdraw_stroke_model_float16.tflite"

with open(tflite_float16_path, "wb") as f:
    f.write(tflite_float16)

print("Saved:", tflite_float16_path)
print("Size MB:", tflite_float16_path.stat().st_size / 1024 / 1024)

Saved artifact at '/tmp/tmptullxzvc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 5), dtype=tf.float32, name='stroke_input')
Output Type:
  TensorSpec(shape=(None, 30), dtype=tf.float32, name=None)
Captures:
  137584927006672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578389776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578389392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578386704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578389200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578388240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578389008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578390160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578388048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578387088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137582578386128: T

In [ ]:
import numpy as np
import tensorflow as tf

TFLITE_PATH = str(MODEL_DIR / "quickdraw_stroke_model_float32.tflite")

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
print(input_details)

print("Output details:")
print(output_details)

sample = x_test[0:1].astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], sample)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]["index"])[0]

top_ids = output.argsort()[-3:][::-1]

print("True label:", CLASSES[y_test[0]])
print("Top 3:")
for idx in top_ids:
    print(CLASSES[idx], float(output[idx]))

Input details:
[{'name': 'serving_default_stroke_input:0', 'index': 0, 'shape': array([  1, 256,   5], dtype=int32), 'shape_signature': array([ -1, 256,   5], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details:
[{'name': 'StatefulPartitionedCall_1:0', 'index': 48, 'shape': array([ 1, 30], dtype=int32), 'shape_signature': array([-1, 30], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
True label: sun
Top 3:
sun 0.9999988079071045
moon 1.197515189232945e-06
clock 3.3116837272473276e-08


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
model_info = {
    "model_name": "quickdraw_stroke_model",
    "input_shape": [1, MAX_LEN, 5],
    "input_dtype": "float32",
    "output_shape": [1, len(CLASSES)],
    "output_dtype": "float32",
    "classes": CLASSES,
    "class_count": len(CLASSES),
    "sequence_format": ["dx", "dy", "pen_down", "pen_up", "end"],
    "dx_dy_scaled_by": 255.0,
}

with open(MODEL_DIR / "model_info.json", "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

print(json.dumps(model_info, indent=2))

{
  "model_name": "quickdraw_stroke_model",
  "input_shape": [
    1,
    256,
    5
  ],
  "input_dtype": "float32",
  "output_shape": [
    1,
    30
  ],
  "output_dtype": "float32",
  "classes": [
    "The Mona Lisa",
    "apple",
    "banana",
    "cat",
    "dog",
    "fish",
    "bird",
    "airplane",
    "car",
    "bicycle",
    "bus",
    "tree",
    "flower",
    "house",
    "chair",
    "table",
    "cup",
    "fork",
    "umbrella",
    "star",
    "moon",
    "sun",
    "cloud",
    "crown",
    "pizza",
    "ice cream",
    "book",
    "clock",
    "eye",
    "face"
  ],
  "class_count": 30,
  "sequence_format": [
    "dx",
    "dy",
    "pen_down",
    "pen_up",
    "end"
  ],
  "dx_dy_scaled_by": 255.0
}


In [ ]:
!cd /content && zip -r quickdraw_stroke_tflite_export.zip quickdraw_stroke_tflite

from google.colab import files
files.download("/content/quickdraw_stroke_tflite_export.zip")

  adding: quickdraw_stroke_tflite/ (stored 0%)
  adding: quickdraw_stroke_tflite/best.keras (deflated 10%)
  adding: quickdraw_stroke_tflite/quickdraw_stroke_model_float16.tflite (deflated 10%)
  adding: quickdraw_stroke_tflite/final.keras (deflated 10%)
  adding: quickdraw_stroke_tflite/labels.json (deflated 53%)
  adding: quickdraw_stroke_tflite/model_info.json (deflated 58%)
  adding: quickdraw_stroke_tflite/quickdraw_stroke_model_float32.tflite (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>